<a href="https://colab.research.google.com/github/andrea-t94/airflow-net/blob/master/research/finetuning/notebooks/01_finetune_no_unsloth_sdpa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetuning Qwen2.5 on Airflow DAGs (without Unsloth, with PyTorch SDPA)

This notebook demonstrates how to fine-tune the **Qwen/Qwen2.5-Coder-1.5B-Instruct** model on a dataset of **Airflow DAGs** using standard HuggingFace libraries (`transformers`, `peft`, `trl`) — without relying on Unsloth or Flash Attention 2.

It uses **PyTorch SDPA** (Scaled Dot-Product Attention) for efficient attention computation. SDPA is built into PyTorch 2.0+ and automatically dispatches to the best available kernel for the hardware.

### Please note
**1. This notebook has been developed and tested on Google Colab with an A100 GPU**, which is the recommended environment for reproduction. Running it in other environments may require modifications to the setup and installation steps.


**2. If you want to fine-tune the model with your own settings**, you will need to modify:
- **`NEW_MODEL_NAME`** in the configuration cell — change the username to your own Hugging Face account (e.g., `your-username/qwen2.5-1.5b-airflow-instruct`)
- **Hugging Face Token** — ensure you have write permissions to push models to your account

## 1. Setup & Installation
We install the necessary libraries for fine-tuning with QLoRA. No Flash Attention package needed — PyTorch SDPA is built-in.

In [2]:
%%capture
import os
import torch

!pip install transformers accelerate peft trl bitsandbytes datasets huggingface_hub

In [3]:
# Verify GPU and PyTorch
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Detected: {gpu_name}")
    major_version, minor_version = torch.cuda.get_device_capability()
    if major_version >= 8:
        print("\u2705 GPU supports bfloat16 (Ampere or newer). Using SDPA attention.")
    else:
        print("\u2139\ufe0f GPU is older than Ampere (e.g., T4). Using float16 with SDPA attention.")
else:
    raise RuntimeError("No GPU detected! Please change runtime type to GPU in 'Runtime > Change runtime type'.")

PyTorch version: 2.10.0+cu128
GPU Detected: NVIDIA A100-SXM4-40GB
✅ GPU supports bfloat16 (Ampere or newer). Using SDPA attention.


## 2. Configuration & Authentication
Log in to Hugging Face to access datasets and push your model.

In [ ]:
from huggingface_hub import login

# Try to get token from Colab secrets, otherwise prompt
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=True)
except:
    print("Please provide your Hugging Face Token (Permissions: Write)")
    login(add_to_git_credential=True)

In [4]:
# Project Configuration
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATASET_NAME = "andrea-t94/airflow-dag-dataset"

# Output Model Name (Change username if needed)
NEW_MODEL_NAME = "andrea-t94/qwen2.5-1.5b-airflow-instruct"

# Training Parameters
MAX_SEQ_LENGTH = 4096 # Fits most DAG files
LOAD_IN_4BIT = True   # Enable 4-bit quantization (QLoRA) to save memory

## 3. Load Model with QLoRA + PyTorch SDPA
We load the model in 4-bit precision using `bitsandbytes` and apply LoRA adapters via `peft`. PyTorch SDPA is used for efficient attention — no extra packages required.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Determine GPU capabilities
major_version, _ = torch.cuda.get_device_capability()
use_bf16 = major_version >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.padding_side = "right"

# Load model with PyTorch SDPA
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=compute_dtype,
)

print(f"\u2705 Model loaded with PyTorch SDPA attention")

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# LoRA config (same as the Unsloth notebook)
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded with PyTorch SDPA attention
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 4. Load & Format Dataset
We specificy a formatting function to apply the ChatML template (which Qwen uses) to our dataset.

In [6]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset(DATASET_NAME)

# Inspect dataset sizes
print(f"Train size: {len(dataset['train'])}")
if 'eval' in dataset: print(f"Eval size:  {len(dataset['eval'])}")

# Format function for ChatML
# The dataset should have a 'messages' column matching standard chat format
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        # Apply chat template but do NOT tokenize yet
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/560k [00:00<?, ?B/s]

data/eval-00000-of-00001.parquet:   0%|          | 0.00/574k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7414 [00:00<?, ? examples/s]

...

Generating eval split:   0%|          | 0/412 [00:00<?, ? examples/s]

Train size: 7414
Eval size:  412


Map:   0%|          | 0/7414 [00:00<?, ? examples/s]

Map:   0%|          | 0/412 [00:00<?, ? examples/s]

Map:   0%|          | 0/412 [00:00<?, ? examples/s]

## 5. Training
Configure the `SFTTrainer`. We use `gradient_accumulation_steps` to simulate a larger batch size.

In [7]:
from trl import SFTTrainer, SFTConfig

# Determine GPU capabilities for dtype selection
major_version, _ = torch.cuda.get_device_capability()
use_bf16 = major_version >= 8

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset.get("eval"),
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = MAX_SEQ_LENGTH,
        dataset_num_proc = 2,
        packing = True, # Set to True to speed up training if sequence len is variable and usually are shorter than max_seq_len
        per_device_train_batch_size = 4,  # Increase if GPU memory allows
        gradient_accumulation_steps = 8,   # effective_batch = per_device_train_batch_size*gradient_accumulation_steps
        max_steps = 10,                   # Set to -1 for full epochs. Using 10 for testing and estimating training time.
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not use_bf16,
        bf16 = use_bf16,
        logging_steps = 1,
        optim = "adamw_8bit",             # Use 8-bit optimizer to save memory
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "steps",
        eval_strategy = "steps",
        eval_steps = 100,
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
    ),
)

Tokenizing train dataset (num_proc=2):   0%|          | 0/7414 [00:00<?, ? examples/s]

Packing train dataset (num_proc=2):   0%|          | 0/7414 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=2):   0%|          | 0/412 [00:00<?, ? examples/s]

Packing eval dataset (num_proc=2):   0%|          | 0/412 [00:00<?, ? examples/s]

In [8]:
# Start Training
import time
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start
print(f"Training took {elapsed/60:.1f} minutes ({elapsed/3600:.1f} hours)")
print(f"Avg time per step: {elapsed/trainer.state.global_step:.1f}s")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


Training took 7.6 minutes (0.1 hours)
Avg time per step: 45.7s


In [9]:
total_steps_full_run = 696  # 7414 samples / 32 effective batch * 3 epochs
print(f"Estimated full run ({total_steps_full_run} steps): {(elapsed/trainer.state.global_step * total_steps_full_run)/3600:.1f} hours")

Estimated full run (696 steps): 8.8 hours


## 6. Save & Push to Hub
We save LoRA adapters and the full merged model, then push to Hugging Face Hub.

In [ ]:
# 1. Save LoRA Adapters only (Small file size, fast)
model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")
model.push_to_hub(f"{NEW_MODEL_NAME}-lora", token=True)
tokenizer.push_to_hub(f"{NEW_MODEL_NAME}-lora", token=True)

In [ ]:
# 2. Save Merged Model (Full model for direct inference)
from peft import AutoPeftModelForCausalLM

print("Merging LoRA weights into base model...")
merged_model = model.merge_and_unload()

# Save locally
merged_model.save_pretrained("merged_model", safe_serialization=True)
tokenizer.save_pretrained("merged_model")

# Push to Hub
print("Pushing merged model to Hub...")
merged_model.push_to_hub(NEW_MODEL_NAME, token=True, safe_serialization=True)
tokenizer.push_to_hub(NEW_MODEL_NAME, token=True)
print(f"Saved merged model to https://huggingface.co/{NEW_MODEL_NAME}")

In [ ]:
# 3. Convert to GGUF (for Ollama/Llama.cpp)
# Without Unsloth, we use llama.cpp's convert script directly
print("Converting to GGUF...")

# Install llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git /tmp/llama_cpp 2>/dev/null || true
!cd /tmp/llama_cpp && pip install -r requirements.txt 2>/dev/null

# Convert to GGUF f16 first
!python /tmp/llama_cpp/convert_hf_to_gguf.py merged_model --outfile merged_model.f16.gguf --outtype f16

# Quantize to Q4_K_M
!cd /tmp/llama_cpp && make -j quantize 2>/dev/null
!/tmp/llama_cpp/llama-quantize merged_model.f16.gguf merged_model.Q4_K_M.gguf Q4_K_M

# Upload GGUF to Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="merged_model.Q4_K_M.gguf",
    path_in_repo="qwen2.5-coder-1.5b-instruct.Q4_K_M.gguf",
    repo_id=NEW_MODEL_NAME,
    token=True
)
print(f"GGUF uploaded to https://huggingface.co/{NEW_MODEL_NAME}")